# [기초-실습] 통계 101×데이터 분석: (4장) 추론통계~신뢰구간

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## ⚙️ 환경 준비 — 한글 폰트 설치 및 라이브러리 불러오기

In [ ]:
# 구글 코랩 환경에서 한글 폰트 설치 및 설정하기
# 필요시 아래 코드 실행 후, [런타임] - [세션 다시 시작] 후 셀을 다시 실행하세요.
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
# 파이썬 라이브러리 및 모듈 가져오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'NanumGothic'  # 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

# 문제 1. 표본조사 체험하기

**📘 문제**

- 온라인 쇼핑몰은 전체 고객 수가 너무 많아, 모든 고객을 조사하기 어렵습니다.

- 그래서 무작위로 고객 30명을 뽑아 평균 만족도를 계산하고 이를 전체 만족도의 추정값으로 사용하려 합니다.

- 이번 실습에서는 직접 표본을 뽑고, 표본 평균을 구해보며,
  **“표본마다 결과가 달라질 수 있다”**는 추론 통계의 핵심 개념을 체험해봅니다.

In [ ]:
# 모집단 생성 (전체 고객 만족도 10,000명)
np.random.seed(2025)
population = np.random.normal(loc=7.5, scale=1.2, size=10000)
population = np.clip(population, 1, 10)  # 1점 ~ 10점 사이로 제한
df_pop = pd.DataFrame({'score': population})

# 전체 모집단 시각화
sns.histplot(df_pop['score'], bins=30, kde=True)
plt.title("전체 고객 만족도 분포 (모집단)")
plt.xlabel("만족도 점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 표본을 무작위로 여러 번 뽑아 보고, 표본 평균이 어떻게 변하는지 확인해봅시다.

- 히스토그램을 그리고, 표본 평균의 분포 형태를 관찰해봅시다.

In [ ]:
# [문제 1] Q1. 모집단에서 무작위로 30명을 뽑아 표본 평균을 구해봅시다.
sample = df_pop.sample(n=30, random_state=2025)
sample_mean = sample["score"].mean()

print(sample_mean)

In [ ]:
# [문제 1] Q2. 이 과정을 500번 반복하고, 표본 평균을 리스트에 저장합니다.
sample_means = []

for i in range(500):
    sample = df_pop.sample(n=30)
    sample_mean = sample["score"].mean()
    sample_means.append(sample_mean)


In [ ]:
# [문제 1] Q3. 표본 평균들의 분포를 히스토그램으로 그려봅시다. 평균선도 함께 표시해 봅시다.
import matplotlib.pyplot as plt
import numpy as np

plt.hist(sample_means, bins="auto", edgecolor="black")

mean_of_sample_means = np.mean(sample_means)

plt.axvline(
    mean_of_sample_means,
    linestyle="--",
    linewidth=2,
    label=f"평균: {mean_of_sample_means:.2f}"
)

plt.xlabel("표본 평균")
plt.ylabel("빈도")
plt.title("표본 평균의 분포")
plt.legend()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 평균들은 어떤 값 주변에 많이 분포해 있나요? 이 값은 전체 모집단 평균과 얼마나 비슷한가요?

- 표본을 1번 뽑았을 때와 500번을 반복해서 뽑았을 때, 표본 평균의 분포나 신뢰성에는 어떤 차이가 있나요

- 친구가 다른 표본을 뽑았다면 같은 평균이 나왔을까요? 비슷한 결과가 나왔더라도 완전히 같지 않았다면, 그 이유는 무엇일까요?

- 표본 평균들의 분포는 어떤 모양인가요? 종 모양의 정규분포처럼 보이나요? 그렇다면 왜 그렇게 되는 걸까요?

In [ ]:
# [문제 1] 데이터를 어떻게 읽을까요?
# 표본 평균들은 대부분 7.5 주변에 분포한다. 이는 처음 설정한 전체 모집단의 평균인 약 7.5와 매우 비슷하다. 각각의 표본 평균은 조금씩 다르지만, 500개의 표본 평균을 전체적으로 보면 모집단 평균 근처에 모이는 것을 확인할 수 있다.

# 표본을 한 번만 뽑으면 우연히 평균이 높거나 낮은 표본이 선택될 수 있기 때문에 그 결과만으로 모집단을 판단하기에는 불안정하다. 반면 표본 추출을 500번 반복하면 표본 평균들이 어떤 범위에 주로 나타나는지 확인할 수 있다. 따라서 한 번의 표본 결과보다 모집단 평균을 더 안정적으로 추정할 수 있다.

# 친구가 모집단에서 다른 표본 30개를 뽑았다면 나와 완전히 같은 평균이 나오지는 않을 가능성이 크다. 무작위 추출을 할 때마다 선택되는 데이터가 달라지기 때문이다. 하지만 같은 모집단에서 같은 크기의 표본을 뽑았으므로 평균은 대체로 7.5 근처에서 비슷하게 나타날 것이다. 이러한 표본 간의 차이를 표본추출 변동이라고 한다.

# 표본 평균들의 히스토그램은 가운데가 높고 양쪽으로 갈수록 낮아지는 종 모양에 가깝게 나타난다. 이는 여러 개의 값을 뽑아 평균을 계산하는 과정을 반복하면, 원래 모집단의 분포와 관계없이 표본 평균의 분포가 정규분포에 가까워지는 중심극한정리 때문이다. 이번 실습의 모집단도 정규분포로 생성되었기 때문에 표본 평균의 분포가 더욱 뚜렷한 정규분포 모양으로 나타난다.


# 문제 2. 중심극한정리

**📘 문제**

- 현실에서는 모집단의 분포가 정규분포가 아닐 수도 있습니다.

- 예를 들어, 일부 고객은 매우 높은 점수를 주고, 대부분은 낮은 점수를 주는 만족도 분포가 있을 수 있죠. (예: 지수분포)

- 이처럼 원래 분포가 비정규분포여도,
  표본을 여러 번 뽑아 평균을 계산하면, 그 평균들의 분포는 정규분포에 가까워진다는 것을
  **중심극한정리(Central Limit Theorem)**라고 합니다.

- 이번 실습에서는 다양한 크기의 표본을 뽑아 평균을 계산하고,
  그 평균들의 분포가 어떻게 변하는지를 직접 실험해 봅니다.

In [ ]:
# 지수분포를 따르는 모집단 생성
np.random.seed(2025)
population = np.random.exponential(scale=50, size=100000)  # 평균 50, 비대칭 분포

# 모집단 시각화
sns.histplot(population, bins=50, kde=True)
plt.title("고객 구매 금액 분포 (모집단: 지수분포)")
plt.xlabel("구매 금액")
plt.show()

**📌 아래를 수행해 보세요:**

- 비대칭적인 모집단(지수분포)에서 무작위로 표본을 추출해 평균을 구해봅시다.

- 표본 크기를 바꿔가며, 표본 평균들의 분포가 어떻게 변화하는지 확인해봅시다.

- 히스토그램을 그리고, 분포의 모양을 관찰해봅시다.

- 표본 크기가 커질수록 표본 평균 분포의 모양과 **퍼진 정도(분산)**가 어떻게 변하는지 관찰해봅시다.

In [ ]:
# [문제 2] Q1. 모집단에서 표본을 1000번 뽑고, 각 표본의 평균을 구해봅시다.
# 표본 크기 = 5일 때
# [문제 2] Q1. 모집단에서 표본을 1000번 뽑고, 각 표본의 평균을 구해봅시다.
# 표본 크기 = 5일 때

sample_means_5 = []

for i in range(1000):
    sample = np.random.choice(population, size=5)
    sample_mean = sample.mean()
    sample_means_5.append(sample_mean)

print(sample_means_5)

In [ ]:
# [문제 2] Q2. 위 과정을 표본 크기 30, 100일 때도 반복해봅시다.
# sample_size = 30, 100

sample_means_30 = []
sample_means_100 = []

for i in range(1000):
    sample_30 = np.random.choice(population, size=30)
    sample_means_30.append(sample_30.mean())

    sample_100 = np.random.choice(population, size=100)
    sample_means_100.append(sample_100.mean())

In [ ]:
# [문제 2] Q3. 각 표본 크기별로 표본 평균들의 분포를 히스토그램으로 그려봅시다.
# 평균선을 함께 표시해 봅시다.

import matplotlib.pyplot as plt
import numpy as np

# 표본 크기 5
plt.hist(sample_means_5, bins=30, edgecolor="black")
mean_5 = np.mean(sample_means_5)
plt.axvline(mean_5, linestyle="--", linewidth=2, label=f"평균: {mean_5:.2f}")
plt.title("표본 크기 5일 때 표본평균의 분포")
plt.xlabel("표본 평균")
plt.ylabel("빈도")
plt.legend()
plt.show()

# 표본 크기 30
plt.hist(sample_means_30, bins=30, edgecolor="black")
mean_30 = np.mean(sample_means_30)
plt.axvline(mean_30, linestyle="--", linewidth=2, label=f"평균: {mean_30:.2f}")
plt.title("표본 크기 30일 때 표본평균의 분포")
plt.xlabel("표본 평균")
plt.ylabel("빈도")
plt.legend()
plt.show()

# 표본 크기 100
plt.hist(sample_means_100, bins=30, edgecolor="black")
mean_100 = np.mean(sample_means_100)
plt.axvline(mean_100, linestyle="--", linewidth=2, label=f"평균: {mean_100:.2f}")
plt.title("표본 크기 100일 때 표본평균의 분포")
plt.xlabel("표본 평균")
plt.ylabel("빈도")
plt.legend()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 크기가 작을 때 (예: 5), 평균들의 분포는 어떤 모양인가요?

- 표본 크기가 커질수록 평균 분포의 모양은 어떤 변화를 보이나요?

- 원래 모집단은 비대칭이었는데, 왜 평균들의 분포는 정규분포처럼 바뀌었을까요?

- 이 실험을 통해 중심극한정리를 어떻게 이해하게 되었나요?

- 표본 크기에 따라 **분포의 넓이(흩어짐)**는 어떻게 달라지나요?

In [ ]:
# [문제 2]데이터를 어떻게 읽을까요?

# 표본 크기가 5처럼 작을 때는 표본평균들의 분포가 아직 오른쪽으로 약간 치우쳐 있고, 넓게 퍼져 있는 모습을 보인다. 원래 모집단이 지수분포처럼 비대칭이기 때문에 그 특징이 어느 정도 남아 있는 것이다.

# 표본 크기가 30, 100으로 커질수록 표본평균들의 분포는 점점 좌우 대칭적인 종 모양에 가까워진다. 또한 표본평균들이 모집단 평균인 약 50 주변에 더 많이 모이는 것을 확인할 수 있다.

# 원래 모집단은 비대칭이지만, 여러 값을 뽑아 평균을 구하는 과정을 반복하면 각각의 값이 가진 높고 낮은 차이가 평균을 내는 과정에서 서로 완화된다. 따라서 표본 크기가 충분히 커질수록 표본평균의 분포는 정규분포에 가까워진다.

# 이 실험을 통해 중심극한정리는 모집단 자체가 정규분포로 바뀐다는 의미가 아니라, 모집단의 분포가 비정규분포이더라도 충분한 크기의 표본을 반복해서 뽑아 계산한 표본평균들의 분포가 정규분포에 가까워진다는 의미임을 이해할 수 있었다.

# 표본 크기가 커질수록 표본평균 분포의 넓이는 좁아지고 흩어짐은 줄어든다. 표본 크기가 작으면 어떤 값이 뽑히느냐에 따라 평균이 크게 달라질 수 있지만, 표본 크기가 커지면 많은 값이 평균에 함께 반영되므로 우연한 영향이 줄어들고 평균이 모집단 평균 주변에 더 안정적으로 모이기 때문이다.


# 문제 3. 표준오차

**📘 문제**

- 앞선 실습에서 우리는 **표본 크기(n)가 커질수록 표본 평균들의 분포가 더 좁아진다**는 것을 확인했습니다.
- 이처럼 표본 평균들이 얼마나 흩어져 있는지(분포의 퍼진 정도)를 나타내는 값을 **표준오차(Standard Error, SE)**라고 부릅니다.
- 표준오차는 **표본 평균들의 표준편차**와 같은 의미이며, 이는 우리가 뽑은 표본 평균이 실제 모평균과 평균적으로 얼마나 떨어져 있을지를 나타내는 **'예상 오차의 크기'**입니다.

- 통계학적으로 이 표준오차는 **`SE = σ / √n`** (모집단 표준편차 / 표본 크기의 제곱근) 이라는 공식으로 계산할 수 있습니다.
- 이 공식은 **표본 크기(n)가 커질수록 표준오차(SE)가 작아진다**는 것을 명확히 보여줍니다.

- 이번 실습에서는 여러 크기의 표본을 뽑아, 시뮬레이션을 통해 얻은 **표본 평균들의 표준편차(실험값)**가 공식으로 계산한 **표준오차(이론값)**와 얼마나 일치하는지 직접 확인해봅니다.

In [ ]:
# 모집단 생성 (평균 100, 표준편차 15)
np.random.seed(2025)
population = np.random.normal(loc=100, scale=15, size=100000)

# 모집단 시각화
sns.histplot(population, bins=40, kde=True)
plt.title("모집단 분포 (평균 100, 표준편차 15)")
plt.xlabel("값")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단에서 여러 크기의 표본(10, 30, 100, 500)을 각각 1000번 뽑고, 그 평균들을 구한 뒤, **표본 평균들의 표준편차(=실험적 표준오차)**를 계산해봅시다.

- 이 결과를 이론적인 표준오차 공식과 비교하는 표를 만들고, 시각화해봅시다.

In [ ]:
# [문제 3] Q1. 표본 크기 10, 30, 100, 500에 대해 각각 1000번 표본을 뽑고, 평균을 구해봅시다.
# 각 표본 평균 분포의 표준편차를 계산해봅시다.
# 결과를 리스트에 저장하고, 표로 정리해봅시다.

sample_sizes = [10, 30, 100, 500]
results = []

for n in sample_sizes:
    sample_means = []

    for i in range(1000):
        sample = np.random.choice(population, size=n)
        sample_means.append(sample.mean())

    sample_mean_std = np.std(sample_means)

    results.append({
        "표본 크기": n,
        "표본평균의 표준편차": sample_mean_std
    })

df_results = pd.DataFrame(results)
df_results

In [ ]:
# [문제 3] Q2. 이론적인 표준오차와 비교해봅시다.
# [공식] 표준오차(SE) = 모집단 표준편차 / √표본크기

population_std = np.std(population)

df_results["이론적 표준오차"] = (
    population_std / np.sqrt(df_results["표본 크기"])
)

df_results["차이"] = (
    df_results["표본평균의 표준편차"]
    - df_results["이론적 표준오차"]
)

df_results

In [ ]:
# [문제 3] Q3. 실험값과 이론값을 시각화해봅시다.
# 표본 크기를 x축, 표준오차를 y축으로 한 꺾은선 그래프를 그려봅시다.

plt.plot(
    df_results["표본 크기"],
    df_results["표본평균의 표준편차"],
    marker="o",
    label="실험값"
)

plt.plot(
    df_results["표본 크기"],
    df_results["이론적 표준오차"],
    marker="o",
    linestyle="--",
    label="이론값"
)

plt.xlabel("표본 크기")
plt.ylabel("표준오차")
plt.title("표본 크기에 따른 표준오차 비교")
plt.legend()
plt.grid()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 크기가 작을수록, 표본 평균의 분포는 어떤 모양인가요? 넓게 퍼져 있나요?

- 표본 크기가 커질수록, 평균 분포는 어떻게 변하나요?

- 실험값과 이론값(공식 계산값)은 얼마나 비슷한가요?

- 왜 표본 크기가 커질수록 표준오차는 작아질까요?

In [ ]:
# [문제 3] 데이터를 어떻게 읽을까요?

# 표본 크기가 작을수록 표본평균의 분포는 넓게 퍼져 있다. 적은 수의 자료만 평균에 반영되므로 어떤 값이 뽑히는지에 따라 표본평균이 크게 달라질 수 있기 때문이다.

# 표본 크기가 커질수록 표본평균의 분포는 모집단 평균인 100 주변으로 더 좁게 모인다. 따라서 표본평균의 변동이 줄어들고 모집단 평균을 더 안정적으로 추정할 수 있다.

# 시뮬레이션으로 구한 표본평균들의 표준편차인 실험값과 표준오차 공식으로 계산한 이론값은 거의 비슷하게 나타난다. 완전히 같지 않은 이유는 무작위 표본 추출에 따른 오차가 존재하고, 실험을 1000번만 반복했기 때문이다.

# 표본 크기가 커질수록 많은 값이 평균에 반영되어 개별적인 극단값이나 우연한 표본 선택의 영향이 줄어든다. 또한 표준오차는 모집단 표준편차를 표본 크기의 제곱근으로 나누어 계산하므로, 표본 크기가 커질수록 표준오차는 작아진다.


# 문제 4. 신뢰구간 계산과 해석

**📘 문제**

- 표본 평균은 모집단 평균을 추정하는 좋은 점 추정(Point Estimation) 값이지만, 표본오차 때문에 정확히 일치하지는 않습니다.

- 그래서 우리는 "모집단 평균이 아마 이 범위 안에 있을 것이다"라고 **구간으로 추정(Interval Estimation)**하는 것이 더 합리적입니다. 이때 사용하는 개념이 바로 **신뢰구간(Confidence Interval)**입니다.

- 신뢰구간은 표본평균 ± 오차범위 형태로 계산되며, 이 오차범위는 신뢰수준(예: 95%, 99%)과 표본오차에 의해 결정됩니다.

- 이번 실습에서는 **모집단 표준편차(σ)를 알 때(z-분포)**와 **모를 때(t-분포)**의 신뢰구간을 각각 계산해보고, 신뢰수준에 따라 구간의 폭이 어떻게 변하는지 확인해봅니다.

In [ ]:
# 모집단 생성
np.random.seed(2025)
population = np.random.normal(loc=70, scale=10, size=10000)

# 모집단 시각화
sns.histplot(population, bins=40, kde=True)
plt.title("모집단 분포 (평균 70, 표준편차 10)")
plt.xlabel("점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단에서 30명을 무작위로 뽑아 평균, 표준편차, 표준오차를 계산해보세요.

- 95% 신뢰구간을 z-분포와 t-분포를 각각 사용해서 계산해보세요.

- 신뢰수준을 바꿨을 때(90%, 99%) 신뢰구간이 어떻게 변하는지 확인해보세요.

In [ ]:
# [문제 4] Q1. 모집단에서 표본 30명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.
# 표준오차도 함께 계산해보세요.

sample = np.random.choice(population, size=30, replace=False)

sample_mean = np.mean(sample)
sample_std = np.std(sample, ddof=1)
standard_error = sample_std / np.sqrt(len(sample))

print("표본 평균:", sample_mean)
print("표본 표준편차:", sample_std)
print("표준오차:", standard_error)

In [ ]:
# [문제 4] Q2. 모집단의 표준편차를 알고 있다고 가정하고, z-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

population_std = 10
z_value = 1.96
n = len(sample)

standard_error_z = population_std / np.sqrt(n)

lower_z = sample_mean - z_value * standard_error_z
upper_z = sample_mean + z_value * standard_error_z

print("95% 신뢰구간:", (lower_z, upper_z))

In [ ]:
# [문제 4] Q3. 모집단의 표준편차를 모른다고 가정하고, 표본 표준편차와 t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

from scipy import stats

n = len(sample)
df = n - 1
t_value = stats.t.ppf(0.975, df)

standard_error_t = sample_std / np.sqrt(n)

lower_t = sample_mean - t_value * standard_error_t
upper_t = sample_mean + t_value * standard_error_t

print("95% 신뢰구간:", (lower_t, upper_t))

In [ ]:
# [문제 4] Q4. 신뢰수준을 90%, 99%로 바꿔가며 신뢰구간을 계산해보고, 그 폭을 비교해봅시다.

confidence_levels = [0.90, 0.95, 0.99]
results_ci = []

for confidence in confidence_levels:
    alpha = 1 - confidence
    t_value = stats.t.ppf(1 - alpha / 2, df)

    margin_error = t_value * standard_error_t
    lower = sample_mean - margin_error
    upper = sample_mean + margin_error
    width = upper - lower

    results_ci.append({
        "신뢰수준": f"{int(confidence * 100)}%",
        "하한": lower,
        "상한": upper,
        "구간 폭": width
    })

df_ci = pd.DataFrame(results_ci)
df_ci

**🧠 데이터를 어떻게 읽을까요?**

- z-분포와 t-분포를 사용한 신뢰구간은 얼마나 차이가 있나요?

- 신뢰수준이 높아질수록 신뢰구간의 폭은 어떻게 변하나요? 왜 그럴까요?

- 신뢰구간이 넓다는 건 좋은 걸까요? 나쁜 걸까요?

- 이 데이터가 실제 고객 만족도라면, 신뢰구간 정보를 마케팅 전략에 어떻게 활용할 수 있을까요?

In [ ]:
# [문제 4] 데이터를 어떻게 읽을까요?

# z-분포랑 t-분포로 계산한 신뢰구간은 엄청 크게 차이나지는 않았다. 다만 t-분포로 구한 구간이 조금 더 넓게 나왔다. 모집단 표준편차를 모르면 표본 표준편차를 사용해야 해서 불확실성이 더 크기 때문이다.

# 신뢰수준이 90%, 95%, 99%로 높아질수록 신뢰구간의 폭은 더 넓어졌다. 모평균이 구간 안에 들어갈 가능성을 더 높이려면 범위를 더 넓게 잡아야 하기 때문이다.

# 신뢰구간이 넓다고 무조건 좋은 것은 아니다. 구간이 넓으면 모평균이 포함될 가능성은 높아지지만, 정확히 어느 정도인지 판단하기는 어려워진다. 반대로 구간이 좁으면 모평균을 더 정확하게 추정했다고 볼 수 있다.

# 이 데이터가 실제 고객 만족도라면 평균 점수만 보는 것보다 신뢰구간을 같이 보는 것이 좋다. 예를 들어 평균 만족도가 70점이어도 신뢰구간의 하한이 낮다면 실제 전체 고객의 만족도는 생각보다 낮을 수도 있다. 그러면 만족도 개선이 필요하다고 판단할 수 있다. 반대로 신뢰구간 전체가 목표 점수보다 높다면 현재 마케팅 전략이나 서비스를 유지할 근거로 사용할 수 있다.


# 문제 5. 미니 프로젝트 - 고객 만족도 신뢰구간 추정

**📘 문제**

- 전체 고객 10,000명을 대상으로 만족도 조사를 하는 것은 시간과 비용이 많이 듭니다.
- 그래서 우리는 무작위로 일부 고객만 조사하여, 전체 고객의 평균 만족도를 추정하려 합니다.

- 이 프로젝트에서는 실제와 같은 상황을 가정하여, 표본을 뽑고 신뢰구간을 계산한 뒤, 이 결과를 바탕으로 마케팅 전략에 어떻게 활용할 수 있을지까지 생각해보는 실습을 진행합니다.

In [ ]:
# 모집단 생성 (고객 만족도 10,000명)
np.random.seed(2025)
population = np.random.normal(loc=7.2, scale=1.0, size=10000)
population = np.clip(population, 1, 10)

# 모집단 시각화
sns.histplot(population, bins=30, kde=True)
plt.title("전체 고객 만족도 분포 (모집단)")
plt.xlabel("만족도 점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단을 생성하고, 거기서 표본을 40명 뽑아 평균을 계산해봅시다.

- 표본 평균과 표준편차를 바탕으로 95% 신뢰구간을 계산해봅시다.

- 히스토그램을 그리고 신뢰구간을 시각화해봅시다.

- 이 결과를 어떻게 해석하고, 마케팅 전략에 활용할 수 있을지 생각해봅시다.

In [ ]:
# [문제 5] Q1. 모집단에서 표본 40명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.

sample = np.random.choice(population, size=40, replace=False)

sample_mean = np.mean(sample)
sample_std = np.std(sample, ddof=1)

print("표본 평균:", sample_mean)
print("표본 표준편차:", sample_std)

In [ ]:
# [문제 5] Q2. 표준오차(SE)를 구하고, t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

from scipy import stats

n = len(sample)

# 표준오차
standard_error = sample_std / np.sqrt(n)

# 자유도와 t 임계값
df = n - 1
t_value = stats.t.ppf(0.975, df)

# 오차범위
margin_error = t_value * standard_error

# 95% 신뢰구간
lower = sample_mean - margin_error
upper = sample_mean + margin_error

print("표준오차:", standard_error)
print("t 임계값:", t_value)
print("95% 신뢰구간:", (lower, upper))

In [ ]:
# [문제 5] Q3. 표본 데이터의 히스토그램을 그리고, 평균 및 신뢰구간을 함께 시각화해봅시다.

plt.figure(figsize=(8, 5))

sns.histplot(sample, bins=10, kde=True)

# 표본 평균
plt.axvline(
    sample_mean,
    linestyle="-",
    linewidth=2,
    label=f"표본 평균: {sample_mean:.2f}"
)

# 신뢰구간 하한과 상한
plt.axvline(
    lower,
    linestyle="--",
    linewidth=2,
    label=f"95% 신뢰구간 하한: {lower:.2f}"
)

plt.axvline(
    upper,
    linestyle="--",
    linewidth=2,
    label=f"95% 신뢰구간 상한: {upper:.2f}"
)

# 신뢰구간 범위 표시
plt.axvspan(lower, upper, alpha=0.2)

plt.title("표본 고객 만족도 분포와 95% 신뢰구간")
plt.xlabel("만족도 점수")
plt.ylabel("고객 수")
plt.legend()
plt.show()

In [ ]:
# [문제 5] Q4. 신뢰구간의 결과에 따라 어떤 구체적인 마케팅 전략을 세울 수 있을까요?

# 신뢰구간을 확인하면 단순히 고객 만족도의 평균이 몇 점인지뿐만 아니라, 현재 만족도를 어느 정도까지 믿고 마케팅에 활용할 수 있는지도 판단할 수 있다.

# 예를 들어 표본평균은 높게 나왔지만 신뢰구간의 하한이 7점 아래라면, 일부 고객의 높은 점수 때문에 평균이 높아 보이는 것일 수도 있다. 이 상태에서 “고객 만족도가 높다”는 메시지로 광고를 확대하는 것은 위험하다. 먼저 만족도가 낮은 고객의 공통점을 찾아 배송, 가격, 고객 응대 등 구체적인 불만 요인을 개선해야 한다.

# 반대로 신뢰구간의 하한도 충분히 높다면 전체 고객의 만족도가 실제로 높은 편일 가능성이 크다. 이 경우에는 신규 고객을 대상으로 무조건 할인하는 것보다 기존 만족 고객의 후기와 추천을 활용하는 전략이 더 효과적일 수 있다. 예를 들어 후기 작성 보상, 친구 추천 쿠폰, 재구매 고객 전용 혜택을 제공해 높은 만족도를 실제 구매와 구전으로 연결할 수 있다.

# 또 만족도 점수가 고객마다 크게 퍼져 있다면 전체 고객에게 같은 마케팅을 적용하기보다 고객을 나눠서 접근해야 한다. 만족도가 높은 고객에게는 충성도와 재구매를 높이는 전략을 사용하고, 만족도가 낮은 고객에게는 할인 광고보다 불편 사항을 묻는 설문이나 서비스 회복 쿠폰을 제공하는 것이 더 적절하다.

# 따라서 신뢰구간은 광고를 바로 확대할지 결정하는 숫자라기보다, 지금 필요한 것이 홍보인지 서비스 개선인지 판단하는 기준으로 활용할 수 있다.


**🧠 데이터를 어떻게 읽을까요?**

- 신뢰구간은 몇 점에서 몇 점 사이인가요?

- 이 구간은 전체 모집단 평균을 포함하고 있나요?

- 이 결과를 바탕으로 고객 만족도가 충분히 높다고 말할 수 있을까요?

- 만약 신뢰구간이 너무 넓게 나왔다면, 그 이유는 무엇이고 어떻게 개선할 수 있을까요?

In [ ]:
# [문제 5] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.